<b> Transform Addresses Data 
1. Create one record for each customer with 2 sets of address columns, 1 for shipping and 1 for billing address
2. Write transformed data to the Silver schema

In [0]:
df_addresses = spark.table('gizmobox.bronze.v_addresses')
display(df_addresses)

<b> Create one record for each customer with both addresses, one for each address_type
> [Documentation for PIVOT clause](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-qry-select-pivot)

In [0]:
from pyspark.sql import functions as f
df_pivoted_addresses = (
    df_addresses
    .groupBy('customer_id')
    .pivot('address_type', ['billing', 'shipping'])
    .agg(
        f.max('address_line_1').alias('address_line_1'),
        f.max('city').alias('city'),
        f.max('state').alias('state'),
        f.max('postcode').alias('postcode')
    )
)
display(df_pivoted_addresses)

<b> 2. Write transformed data to the Silver schema 


In [0]:
df_pivoted_addresses.writeTo('gizmobox.silver.py_addresses').createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_addresses